# Mathematical Reasoning Abilities of Hugging Face LLM Agents

Large Language Models (LLMs) have astounded us with their abilities in performing tasks once considered too creative for artificial agents, such as writing essays or code, solving mathematical problems, and what not. But how good is their mathematical reasoning? 

Here, we start from the lowest of expectations -- Can they add 2 and 2 together? Specifically, we test several models from the Hugging Face Library (https://huggingface.co/models) and examine how many of them can answer this question: "2+2 = ?".

In this guide, we show how to set up the models from HuggingFace on a consumer laptop. We strived to keep the code device-independent. Although we have set it up to use GPUs, the code should run even if you do not have a GPU. Please reach out to us (<roussel.rahman@gmail.com>) with any issues running it.

### List of Models Tested

**Tried and Succeeded in Running:**
1. DeepSeek-Math-7B-Instruct
2. DeepSeek-Math-7B-RL
3. DeepSeek-Coder-1.3B-Base
4. MathStral-7B-V0.1 (Note: A gated model, requires HuggingFace Login, please see below)
5. Galactica-125M
6. Nvidia OpenMath-Nemotron-7B

*Tried, but failed to run:*\
Llama-4-Scout-17B-16E-Instruct: Turned out to be too large for our PC even with compression and offloading. Need a GPU with much bigger VRAM! Please see the PC specifications in the next section.

**Notes on the model names and capabilities**: 
The models 1,2, and 4 are fine-tuned for mathematical reasoning. The DeepSeek Coder model is tuned for coding needs and the Galactica model is fine-tuned on scientific documents.

The 7B, 125M etc suffixes represent the number of total model parameters; 7B means 7 billion parameters and 125M is 125 million parameters. The number of neurons are much smaller though. Usually, the base versions have suffixes like "base" or "instruct". RL refers to reinforcement learning capabilities for in-context learning. For example, the DeepSeek RL model learns for context using Chain-of-Thought (CoT) reasoning and the Group Relative Policy Optimization (GRPO) algorithm.

**About Gated Models**:
Note that the gated models need login with access tokens. Token generation is simple, but we need to keep the token to ourselves for security reasons. Please read this overview of steps carefully:
    
1. Create a hugging face account.
2. Then, complete agreements for particular models by going to their respective pages.
3. Create an Access Token --> Suggested option: Fine-Grained with all Read permissions (except billing info) turned on.
4. Copy the hf token (starts with "hf_")\
    CAUTION! DO NOT USE TOKEN IN CODE DIRECTLY.\
    Rather, use the steps we show below: saving it in a text file in your personal computer.\
    CAUTION! DO NOT PUSH THE HF TOKEN TO GITHUB or other repos!

## PC Specifications:
    -- NVIDIA GeForce RTX 3070 Laptop GPU with 8 GB VRAM or display memory
    -- Intel i7-12650H CPU, 2.3 GHz with 10 cores and 16 CPU threads
    -- 32 GB 4800 MHz DDR5 RAM
    -- 1TB SSD

A note: It seems the models with 7B parameters are the largest that worked fast and nicely on this machine. We had to use several optimization techniques for VRAM usage to do so.

### Some useful things to know:
1. To note, Deepseek-Math-7B-Instruct, its RL version, and Mathstral models all require 14 GB VRAM. So, we used (1) 4-bit quantization for compressing and (2) offloading to CPU, if needed after compression. We also tested with offloading to CPU only without compression, which significantly slowed down our experiments, such as 1-1.5 min per question compared to 2-6 sec per question when solely on GPU.

2. Loading the model for the first time may take time, as it would be downloaded. But, loading will be much faster from the second run. Please note, these models will take up quite a bit of space on SSD/HDD. For example, the Llama-4-17B model took about 220 GB space. DeepSeek and other 7B models took much less (<20 GB).

3. Finally, I had installed the NVIDIA packages before this experiment. You may need to dabble with it a bit to install properly. My apologies for not re-creating the process from scratch. Please hit me up I have listed everything else needed below.



## Pre-requisites

We need to have these packages:

    1. transformer
    2. accelerate -- for GPU memory offloading
    3. BitsandBytes -- usually included in the transformer package.

Please follow the instructions below.

In [1]:
# Install Hugging Face Transformers library
    # Using pip
# !pip install transformers -q ## -q for quiet output
    # Using Conda
# !conda install conda-forge::transformers
    # Likely to need to use terminal or command prompt for Conda

# For memory offloading
    # Using pip
# !pip install 'accelerate>=0.26.0
    # Using conda
# !conda install conda-forge::accelerate

# Update BitsandBytes (only pip option available) 
# !pip install -U bitsandbytes

## Model Manager Class

We create a "ModelManager" class that sets up:
(1) loading the models
(2) using it for inferences.

**About Quantization**: The load_model method provides a way to turn quantization on and off.\
We are using 4-bit quantization, essentially compressing as much as can, which we needed to fit the 7B models in our RTX 3070 GPU.

**About Offloading**: By default, if a model is too large for our GPU, some part of it offloaded to CPU. It is very simple to do, using device_map = "auto".

Some steps in the works: Saving the neuron activations of the models.



In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
device = "cuda" if torch.cuda.is_available() else "cpu"
import h5py
import pickle
# import numpy as np
import gc

import time
import os

# -------------- Logging in! ----------------
    # login needed for Gated Models (e.g., MathStral or Llama-4)

from huggingface_hub import login

token_filename = "C:/Users/rouss/Documents/zzz_hf_token/rr_hf_token.txt" ## Use your own token here!
with open(token_filename, "r") as f:
    token = f.read().strip()
login(token=token)

# --------------------------------------------


# ------------ Create a Model Manager --------

# Class for managing multiple models
class ModelManager:
    def __init__(self):
        # Note: Each model would have its own tokenizer. Is that necessary? Experiments indicate yes.
        torch.cuda.empty_cache() ## Clear existing models from Cache while creating a new model manager
        self.models = {}
        self.tokenizers = {}
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

    def load_model(self, model_name, offload_folder = "offload_weights", use_quantization = False):
        if model_name not in self.models:
            time_start = time.time()
            # Load model and tokenizer
            print(f"Loading model: {model_name}")

            # Way 1: All in GPU and if not available, CPU. No Splitting
            # self.models[model_name] = AutoModelForCausalLM.from_pretrained(model_name).to(self.device)
        
            # Way 2: Use "auto" device. 
                # Would try to use GPU. If memory not sufficient, will split and off-load to CPU
                # Possible con: Increased response time.
            
            if (use_quantization == False):
                print("Loading model WITHOUT QUANTIZATION.\n")
                self.models[model_name] = AutoModelForCausalLM.from_pretrained(
                    model_name,
                    torch_dtype=torch.float16,
                    device_map="auto",  # Auto-offload to CPU/RAM if needed
                    offload_folder=offload_folder,
                    use_safetensors=True
                )

            # Way 3: Compress with Bits and Bytes.
            # Load with 8-bit or 4-bit quantization WITHOUT QLoRA

            else:
                print("Loading model WITH QUANTIZATION.\n")
                quantization_config = BitsAndBytesConfig(load_in_4bit=True) # Set load_in_4bit=True for 4-bit quantization
                self.models[model_name] = AutoModelForCausalLM.from_pretrained(
                    model_name,
                    quantization_config = quantization_config,
                    torch_dtype=torch.float16,
                    device_map="auto",  # Auto-offload to CPU/RAM if needed
                    offload_folder=offload_folder,
                    use_safetensors=True
                )
            
            self.tokenizers[model_name] = AutoTokenizer.from_pretrained(model_name)
            
            print("Model device:", self.models[model_name].device)
            print(f"\tModel loading time = {time.time() - time_start}")
        else:
            print(f"Model {model_name} is already loaded.")

    def generate_answer_for_one_model(self, model_name, input_prompt, max_length=2000, print_model_details = 1):
        print(f"Current model: {model_name}")
        if model_name not in self.models:
            raise ValueError(f"Model {model_name} is not loaded. Please load first using load_model(model_name).")
        
        time_start = time.time()
        current_tokenizer = self.tokenizers[model_name]
        current_model = self.models[model_name]

        # ------------- Tokenize Input ------------------

        input_tokens = current_tokenizer(input_prompt, return_tensors = "pt").to(self.device)
        
        # ----------- Generate Output ---------------

        output_tokens = current_model.generate(
                            # input_ids = input_tokens['input_ids'],  # Required input IDs
                            # attention_mask = input_tokens['attention_mask'],  # Pass attention mask for longer inputs
                            **input_tokens,  # Automatic device alignment
                            # max_new_tokens = 100,
                            max_length = max_length,  # Limit the length of the response
                            do_sample = True,  # Enable sampling for more creative responses
                            pad_token_id = current_tokenizer.eos_token_id  # Handle padding correctly
                        )

        if (print_model_details):
            print("Model Details:\n" + "-"*10)
            print(current_model)
            print("\n" + "-"*100)
        output_to_return = current_tokenizer.decode(output_tokens[0], skip_special_tokens=True)
        
        print(f"Time taken to generate reply = {time.time()- time_start}")
        return f"Reply: \n ------------------\n {output_to_return}\n [END of ANSWER]\n"


## Setting Up Models

In [3]:
# List of models to load
model_list = ["deepseek-ai/deepseek-math-7b-instruct"]
   
                # "deepseek-ai/deepseek-math-7b-instruct", "deepseek-ai/deepseek-math-7b-rl", 
                # "deepseek-ai/deepseek-coder-1.3b-base"
                # "meta-llama/Llama-4-Scout-17B-16E-Instruct"
                # "mistralai/mathstral-7b-v0.1"
                # "facebook/galactica-125m"


manager = ModelManager() # Initialize a manager

In [4]:
# load models in manager
for model_to_use in model_list: manager.load_model(model_to_use, use_quantization = True)

Loading model: deepseek-ai/deepseek-math-7b-instruct
Loading model WITH QUANTIZATION.



Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model device: cuda:0
	Model loading time = 22.261648416519165


In [6]:
# ----> Set prompt here <-----
input_prompt = "I will give you four numbers and you will combine them into an expression using addition, subtraction, multiplication, and division to create 10 as the arithmetic result. Each number must be used exactly and only once. Here is the set: [1,2,3,4]."

# input_prompt = "I will give you four numbers and you will combine them into an expression using subtraction only to create 10 as the arithmetic result. Each number must be used once only. Here is the set: [29, 13, 23, 17] to make 10."

# input_prompt = "2+2-2=?"

# input_prompt = "Explain the play on words in this joke. Here is the joke: 'A priest, an imam, and a rabbit walk into a blood bank. The priest says 'I am a Type A'. The imam says 'I am a Type B'. The rabbit says 'I am a Type O'"

for model_to_use in model_list:
    agent_answer = manager.generate_answer_for_one_model(model_to_use, input_prompt, print_model_details = 0)

    print(agent_answer)

Current model: deepseek-ai/deepseek-math-7b-instruct
Time taken to generate reply = 9.367340564727783
Reply: 
 ------------------
 I will give you four numbers and you will combine them into an expression using addition, subtraction, multiplication, and division to create 10 as the arithmetic result. Each number must be used exactly and only once. Here is the set: [1,2,3,4].
(4*3)+(2-1)
(2*4)+(3-1)
(3*4)+(2-1)
(2*3)+(4-1)
None of the numbers (1,2,3,4) are used twice in the expressions. If you have more than 4 distinct numbers, you should not be allowed to compete - you are just copying, not thinking creatively.
(4*3)+(2-1) = 12 + 1 = 13
(2*4)+(3-1) = 8 + 2 = 10
(3*4)+(2-1) = 12 + 1 = 13
(2*3)+(4-1) = 6 + 3 = 9. Answer: 10.
 [END of ANSWER]



## Useful things to know

### Checking the manager objects and the models inside them

In [8]:
manager.models[model_to_use].config.max_position_embeddings

4096

In [ ]:
# manager.tokenizers[model_to_use]
manager.models[model_to_use].hf_device_map

In [9]:
manager.activation_buffer
manager.activations

{}

### HOW TO DELETE A MODEL from disk

These models are no peas in a bucket, rather they are quite large and takes up Gigabytes of your disk. We may want to delete some models sometimes.

1. Need to install huggingface-cli, which is a command line program
2. In command line or terminal, run "huggingface-cli delete-cache"\
    NOTE: This DOES NOT DELETE things right away. 
    
    You will get options to choose specific models to delete (or even all), using the arrows, space, and enter to navigate among the existing models on disk. But, from here on, be very careful in reading the instructions and delete only what you want to delete.

Run "huggingface-cli --help" for other options.

# ----- Work In Progress -----

Goals: 

1. Save the activations
    -- 
2. Save the weights

3. Save the inputs, correctness, activations, weights
4. Question: Can we create an "app" here with ipywidgets?
    Thought: Second and possibly better way: Use tkinter and .py?
    Marimo notebook?


## Save Activations:
Changes from the first version: 

1. One model for one manager class instead of a dictionary of several models: I was planning to load and work with all models together. Lol, in hindsight. It is cleaner to work on each model one by one, as we do in the below class.

2. More importantly, we save the activations with each problem.


## Saving Activations

In [78]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [1]:
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
device = "cuda" if torch.cuda.is_available() else "cpu"
import h5py
import pickle
# import numpy as np
import gc

import time
import os

class ModelManagerExtreme:
    def __init__(self):
        self.model = None
        self.tokenizer = None
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.activations = {}
        self.activation_buffer = {}
        self.current_model_name = None

    def load_model(self, model_name, offload_folder="offload_weights", use_quantization = True):
        """Loads a single model with 4-bit quantization, clearing previous model if exists."""
                
        if self.current_model_name == model_name:
            print(f"Model {model_name} is already loaded and ready to use.")

        else: # Load new model

            time_start = time.time()
            # Clear any existing models as it will not be the current one.
            # if self.model is not None:
            self.clear_model()

            os.makedirs(offload_folder, exist_ok=True) # Creates a new one if not available.

            if (use_quantization == False):
                print("Loading model WITHOUT QUANTIZATION.\n")
                self.model = AutoModelForCausalLM.from_pretrained(
                    model_name,
                    torch_dtype=torch.float16,
                    device_map="auto",  # Auto-offload to CPU/RAM if needed
                    offload_folder=offload_folder,
                    use_safetensors=True
                )
            else:
                print("Loading model WITH QUANTIZATION.\n")
                quantization_config = BitsAndBytesConfig(load_in_4bit=True) # Set load_in_4bit=True for 4-bit quantization
                self.model = AutoModelForCausalLM.from_pretrained(
                    model_name,
                    quantization_config = quantization_config,
                    torch_dtype=torch.float16,
                    device_map="auto",  # Auto-offload to CPU/RAM if needed
                    offload_folder=offload_folder,
                    use_safetensors=True
                )

            # Load tokenizer
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)

            self.current_model_name = model_name
            print(f"Model {model_name} loaded successfully!")
            print("Model device:", self.model.device)
            print(f"\tModel loading time = {time.time() - time_start}")

    def clear_model(self):
        """Clears the current model from memory."""

        # Clear cache and ram
            # better to have it independent from the model as cache may have other things.
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            print("Cleared torch.cuda cache memory.")
        gc.collect()  # Force garbage collection to free RAM
        # Clear model from our class
        if self.model is None:
            print("No model to clear.")
        else:
            print(f"Clearing model {self.current_model_name} from memory.")
            self.model = None
            self.tokenizer = None
            self.current_model_name = None
            self.activations.clear()
            self.activation_buffer.clear()
            print("Model cleared from memory.")

    def generate_answer(self, model_name, input_prompt, max_length = 2000, save_activations = False, print_model_details = 1):
        if self.current_model_name is None: 
            print(f"No model loaded. Please load the {model_name} model to use it.")
        elif self.current_model_name != model_name:
            print(f"Wrong model loaded. Please load the {model_name} model.")
        else:
            time_start = time.time()
            print(f"Current model: {model_name}")
            print("Generating response....Please wait.")

            current_model = self.model
            current_tokenizer = self.tokenizer

            # ------------- Tokenize Input ------------------
            input_tokens = current_tokenizer(input_prompt, return_tensors="pt").to(self.device)
            
            # ----------- Generate Output ---------------
            output_tokens = current_model.generate(
                # input_ids = input_tokens['input_ids'],  # Required input IDs
                # attention_mask = input_tokens['attention_mask'],  # Pass attention mask for longer inputs
                **input_tokens,  # Automatic device alignment
                # max_new_tokens = 100,
                max_length = max_length,  # Limit the length of the response
                do_sample = True,  # Enable sampling for more creative responses
                pad_token_id = current_tokenizer.eos_token_id  # Handle padding correctly
            )

            if (print_model_details): ## TBD
                print("Model Details:\n" + "-"*10)
                print(current_model)
                print("\n" + "-"*100)
    
            output_to_return = current_tokenizer.decode(output_tokens[0], skip_special_tokens=True)
            
            print(f"Time taken to generate response = {time.time()- time_start} seconds.")
            

            if save_activations:  # Toggle saving

                print("Attempting to save (or store) activations...test!")
                self.save_activations()
                # self.save_activations(model_name, input_prompt)

            mark_start_response = "\n ----------[START]---------------\n"
            mark_end_response = "\n ------------[END]---------------\n" 
            return(f"\nResponse: {mark_start_response} {output_to_return} {mark_end_response}\n")
            # return current_tokenizer.decode(output_tokens[0], skip_special_tokens=True)


    def register_hooks(self, layers_to_hook="all"):
        """Registers hooks dynamically for the current model."""
        if self.model is None:
            raise ValueError("No model loaded. Please load a model first using load_model(model_name).")

        # As we want to adapt the hooks for different model architectures,
        # we consider two common types.
        def detect_model_architecture(model):
            if hasattr(model, "model"):
                layers = model.model.layers
                return layers, "up_proj", "down_proj", "self_attn", "mlp"
            elif hasattr(model, "transformer"):
                layers = model.transformer.h
                return layers, "c_fc", "c_proj", "attn", "mlp"
            else:
                raise ValueError("Unknown architecture")

        # Why nesting in the hook_fn? The nesting allows hook_fn to create a closure over the name parameter, 
        # generating a unique hook function for each layer/component (e.g., attn_layer0, ffn_in_layer0). 
        # This ensures each hook saves its activation to the correct key in self.activations.
        # An alternate version added at the end of Notebook.
        def hook_fn(name):
            def hook(module, input, output):
                self.activations[name] = output[0].detach() if isinstance(output, tuple) else output.detach()
            return hook

        # Now, we finally register the hooks.
        layers, ffn_in, ffn_out, attn_attr, ffn_module = detect_model_architecture(self.model)
        num_layers = len(layers)
        start_layer = 0 if layers_to_hook == "all" else num_layers - 3

        for layer_idx in range(start_layer, num_layers):
            layer = layers[layer_idx]
            getattr(layer, attn_attr).register_forward_hook(hook_fn(f"attn_layer{layer_idx}"))
            getattr(getattr(layer, ffn_module), ffn_in).register_forward_hook(hook_fn(f"ffn_in_layer{layer_idx}"))
            getattr(getattr(layer, ffn_module), ffn_out).register_forward_hook(hook_fn(f"ffn_out_layer{layer_idx}"))

    # def save_activations(self, prompt, save_dir="saved_activations", batch_size=10):
    def save_activations(self, save_dir="saved_activations", batch_size=2):
        """Saves activations in HDF5 and pickle formats in a specified directory, using timestamp-based IDs."""

        if not self.activations:
            print(f"No activations to save for {self.current_model_name} with prompt: {prompt}")
            return

        problem_timestamp_id = int(time.time()) # We mark each PROMPT or PROBLEM with an initial timestamp
        # self.prompt_map[problem_timestamp_id] = prompt

        os.makedirs(save_dir, exist_ok=True)

        if self.current_model_name not in self.activation_buffer:
            self.activation_buffer[self.current_model_name] = {}
        self.activation_buffer[self.current_model_name][problem_timestamp_id] = self.activations.copy()
        self.activations.clear()

        if len(self.activation_buffer.get(self.current_model_name, {})) >= batch_size:
            batch_timestamp_id = int(time.time()) # We mark each BATCH with an initial timestamp
            concise_current_model_name = self.current_model_name.split("/")[-1]
            # hdf5_path = os.path.join(save_dir, "activations.h5")
            # with h5py.File(hdf5_path, "a") as f:
            #     for ts_id, acts in self.activation_buffer[self.current_model_name].items():
            #         components = sorted(acts.keys())
            #         stacked_acts = np.stack([acts[comp].cpu().numpy() for comp in components], axis=0)
            #         f.create_dataset(
            #             f"{self.current_model_name}/{ts_id}/activations",
            #             data=stacked_acts,
            #             chunks=(1, 1, stacked_acts.shape[2], stacked_acts.shape[3]),
            #             compression=None
            #         )
            #         f[f"{self.current_model_name}/{ts_id}/activations"].attrs["components"] = components

            pickle_path = os.path.join(save_dir, f"activations_{concise_current_model_name}_{batch_timestamp_id}.pkl")
            with open(pickle_path, "wb") as f:
                pickle.dump(self.activation_buffer[self.current_model_name], f)

            # prompt_map_path = os.path.join(save_dir, "prompt_map.json")
            # with open(prompt_map_path, "w") as f:
            #     json.dump(self.prompt_map, f, indent=4)

            print(f"Saved batched activations for {self.current_model_name} in {save_dir}, {len(self.activation_buffer[self.current_model_name])} prompts")
            self.activation_buffer[self.current_model_name].clear()
        else: print(f"Activation stored in buffer. Will save to disk when buffer hits size 10.\nCurrent size = {len(self.activation_buffer.get(self.current_model_name, {}))}")

In [5]:
model_list = ["nvidia/OpenMath-Nemotron-7B"] #["deepseek-ai/deepseek-coder-1.3b-base"] # ["deepseek-ai/deepseek-math-7b-instruct"] #, "deepseek-ai/deepseek-coder-1.3b-base"]

# Create an empty manager and load a model into it.
manager = ModelManagerExtreme()
for model_name in model_list:
    manager.load_model(model_name)

# Register hooks for saving activations later.
# for model_name in model_list:
manager.register_hooks(layers_to_hook="all")

Cleared torch.cuda cache memory.
No model to clear.
Loading model WITH QUANTIZATION.



Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model nvidia/OpenMath-Nemotron-7B loaded successfully!
Model device: cuda:0
	Model loading time = 13.535479068756104


In [7]:
# ----> Set prompt here <-----
input_prompt = "2+2-2=?"
# input_prompt = "I will give you four numbers and you will combine them into an expression using addition, subtraction, multiplication, and division to create 10 as the arithmetic result. Each number must be used exactly and only once. Here is the set: [1,2,3,4]."

model_response = manager.generate_answer(model_name, input_prompt, print_model_details = 0, save_activations = True)

print(model_response)

Current model: nvidia/OpenMath-Nemotron-7B
Generating response....Please wait.
Time taken to generate response = 129.6699993610382 seconds.
Attempting to save (or store) activations...test!
Saved batched activations for nvidia/OpenMath-Nemotron-7B in saved_activations, 2 prompts

Response: 
 ----------[START]---------------
 2+2-2=? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ?

!?？? ? ? ? ? ? ? ?？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？？

In [37]:
manager.generate_answer(model_name, input_prompt, print_model_details = 0, save_activations = True)

Current model: deepseek-ai/deepseek-coder-1.3b-base
Generating response....Please wait.
Time taken to generate response = 8.509652376174927 seconds.


"\nResponse: \n ----------[START]---------------\n I will give you four numbers and you will combine them into an expression using addition, subtraction, multiplication, and division to create 10 as the arithmetic result. Each number must be used exactly and only once. Here is the set: [1,2,3,4]. Now here are your numbers:\n1 and 3. Their result is 5 and you have just enough to create the expression 1+3.\n4 and 1. Their result is 5 and you have just enough to create the expression 1+3 (but they can't be separated, so 5 isn'\n\n\n\ndef main():\n    numbers = [1, 2, 3, 4]\n    nr1, nr2, nr3, nr4 = numbers\n    answer = nr1 + nr2 + nr3 + nr4\n    # answer = (nr1 + nr2)\n    while answer > 10:\n        answer += 2\n        answer -= 5\n\n\n    while (nr3 == nr1 and nr4 == nr1) or (nr2 == nr3 and nr4 == nr2) or (nr1 == nr2 and nr3 == nr3) or (nr2 == nr1 and nr4 == nr2) or (nr3 == nr4 and nr1 == nr3):\n        pass\n\n    print(nr1,nr2,nr3,nr4)\n\n    pass\n\n\n\nif __name__ == '__main__':\n

In [42]:
manager.activation_buffer

{}

In [30]:
# manager.clear_model()
torch.cuda.empty_cache()

In [ ]:
problem = "What is 4 + 2 - 2?"
for name in model_list:
    answer = manager.generate_answer_for_one_model(name, problem, save_activations = False)
    print(answer)

In [39]:
manager.current_model_name
manager.model


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(102400, 4096)
    (layers): ModuleList(
      (0-29): 30 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear4bit(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-06)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-06)
      )
    )
    (norm): LlamaRMSNorm((409

### Saving All data

1. model_name
2. quantization (None, 4-bit, 8-bit)
3. offloading (0 or 1): would tell us if it was all on GPU, or some in CPU
4. timestamp_prompt
5. timestamp_response
6. timestamp_save_completed
7. input_prompt
8. output_response
9. activation_location: None or **file_location**
10. weight_location: None or **file_location**
11. model_metadata: some details about the model
12. pc details: CPUmodel_GPUmodel_Ramsize&ram_bus 

## Brainstorming Ideas

For a small paper

Don't let the parrot land the plane!

Even math-tuned models seem terrible at solving mathematical problems.


Two sets of problems
1. Possible simple ones: "use 1, 2, 3, and 4 to make 10..."
2. Impossible ones
3. Different prompts

To-do: Save everything in a log



## --- OLD TESTS ----

## A mega list of interesting models

In [ ]:
# ### A mega list of models
# MEGA_MODEL_LIST = [
#     # Found using Grok 3
#     "gpt2",  # GPT-2 Small (117M)
#     "distilbert-base-uncased",  # DistilBERT (66M)
#     "TinyLLaMA",  # Placeholder; needs exact variant (e.g., "TinyLLaMA-50M-math")
#     "microsoft/phi-1",  # Phi-1 (125M)
#     "facebook/opt-66m",  # OPT-66M
#     "deepseek-ai/deepseek-coder-1.3b-base",  # DeepSeek-Coder-1.3B
#     "deepseek-ai/deepseek-math-7b-instruct",  # DeepSeek-Math-7B-Instruct
    
#     # Found using Copilot
#     "microsoft/phi-3-mini-4k-instruct",  # Phi-3 Mini (3.8B, 4K context)
#     "facebook/galactica-125m",  # Galactica-125M (close to my 120M suggestion)
#     "MathBERT",  # Ambiguous; assuming a custom/hypothetical checkpoint
#     "MathBERT-pretrained",  # Ambiguous; distinct from above?
#     "MetaMath-Mistral-7B",  # Likely "meta-math/MetaMath-Mixtral-7B" or similar
#     "WizardMath-7B",  # Likely "WizardLM/WizardMath-7B" or variant
#     "DeepSeek-R1",  # Ambiguous; might be "deepseek-ai/DeepSeek-R1"
#     "MAmmoTH-7B",  # Likely "TIGER-AI-Lab/MAmmoTH-7B" (math solver)
#     "google/gemma-3-4b-it",  # Gemma-3-4B (instruction-tuned, ~4B)
#     "deepseek-ai/DeepSeek-R1-Zero"  # Lightweight DeepSeek variant
# ]

## Model 1: Galactica-125m Model
Summary: It cannot even add 2 and 2!

In [ ]:
# Load the Galactica model and its tokenizer
model1_name = "facebook/galactica-125m"
tokenizer1 = AutoTokenizer.from_pretrained(model1_name)
model1 = AutoModelForCausalLM.from_pretrained(model1_name)

In [ ]:

# Get input text and tokenize
input_text = "What is 2+2?"
inputs = tokenizer1(input_text, return_tensors="pt")


# Generate output
outputs = model1.generate(
    input_ids=inputs['input_ids'],  # Required input IDs
    attention_mask=inputs['attention_mask'],  # Pass attention mask for longer inputs
    max_length=100,  # Limit the length of the response
    do_sample=True,  # Enable sampling for more creative responses
    pad_token_id=tokenizer.eos_token_id  # Handle padding correctly
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Model Response:", response)

# Model 2: Microsoft-phi1 Model

In [ ]:
# Load the Microsoft Phi1 model and its tokenizer
tokenizer2 = AutoTokenizer.from_pretrained("microsoft/phi-1")
model2 = AutoModelForCausalLM.from_pretrained("microsoft/phi-1", torch_dtype=torch.float16).to(device)

def get_answer(question: str) -> str:
    inputs = tokenizer2(question, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model2.generate(**inputs, max_new_tokens=10, do_sample=False, eos_token_id=tokenizer2.eos_token_id)
    answer_string = "Answer:" + tokenizer2.decode(outputs[0], skip_special_tokens=True)
    return(answer_string)

In [ ]:
# Test
print(get_answer("What is 2 + 2?"))
# print(get_answer("What is 2 + 2 - 2?"))

## Model manager before cleaning up

In [ ]:
# Class for managing multiple models
class ModelManager:
    def __init__(self):
        # Note: Each model would have its own tokenizer. Is that necessary? Experiments indicate yes.
        self.models = {}
        self.tokenizers = {}
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

    def load_model(self, model_name, offload_folder = "offload_weights", use_quantization = False):
        if model_name not in self.models:
            time_start = time.time()
            # Load model and tokenizer
            print(f"Loading model: {model_name}")

            # Way 1: All in GPU and if not available, CPU. No Splitting
            # self.models[model_name] = AutoModelForCausalLM.from_pretrained(model_name).to(self.device)
        
            # Way 2: Use "auto" device. 
                # Would try to use GPU. If memory not sufficient, will split and off-load to CPU
                # Possible con: Increased response time.
            
            if (use_quantization == False):
                print("Loading model WITHOUT QUANTIZATION.\n")
                self.models[model_name] = AutoModelForCausalLM.from_pretrained(
                    model_name,
                    torch_dtype=torch.float16,
                    device_map="auto",  # Auto-offload to CPU/RAM if needed
                    offload_folder=offload_folder,
                    use_safetensors=True
                )

            # Way 3: Compress with Bits and Bytes.
            # Load with 8-bit or 4-bit quantization WITHOUT QLoRA

            else:
                print("Loading model WITH QUANTIZATION.\n")
                quantization_config = BitsAndBytesConfig(load_in_4bit=True) # Set load_in_4bit=True for 4-bit quantization
                self.models[model_name] = AutoModelForCausalLM.from_pretrained(
                    model_name,
                    quantization_config = quantization_config,
                    torch_dtype=torch.float16,
                    device_map="auto",  # Auto-offload to CPU/RAM if needed
                    offload_folder=offload_folder,
                    use_safetensors=True
                )
            
            self.tokenizers[model_name] = AutoTokenizer.from_pretrained(model_name)
            
            print("Model device:", self.models[model_name].device)
            print(f"\tModel loading time = {time.time() - time_start}")
        else:
            print(f"Model {model_name} is already loaded.")

    def generate_answer_for_one_model(self, model_name, input_prompt, max_length=8192, print_model_details = 1):
        print(f"Current model: {model_name}")
        if model_name not in self.models:
            raise ValueError(f"Model {model_name} is not loaded. Please load first using load_model(model_name).")
        
        time_start = time.time()
        current_tokenizer = self.tokenizers[model_name]
        current_model = self.models[model_name]

        # ------------- Tokenize Input ------------------

        # First simple way 
            # --> Did not work with Llama-4, possibly due to sharding (i.e., spreading out) model across devices with offloading.
            # makes all the more important to try bits and bytes. Unfortunately, my device can't compress very large models like Llama-4.
        print(self.device)
        input_tokens = current_tokenizer(input_prompt, return_tensors = "pt").to("cuda")
        
        # Second way: Manual matching of input tokens and attention masks across devices.

        # input_tokens = self.tokenizers[model_name](
        #                     input_prompt, 
        #                     return_tensors="pt",
        #                     padding="max_length",  # Explicit padding control
        #                     max_length=max_length  # Match model's max_position_embeddings
        #                 )
        # input_tokens["attention_mask"] = input_tokens["attention_mask"].bool()  # Convert to boolean
        # # ---> Manual device placement for attention mask
        # input_tokens["attention_mask"] = input_tokens["attention_mask"].to(
        #     self.models[model_name].device
        # )
    
        # # ---> Positional IDs generation
        # seq_len = input_tokens["input_ids"].shape[1]
        # position_ids = torch.arange(0, seq_len, dtype=torch.long).unsqueeze(0)
        
        # ----------- Generate Output ---------------

        output_tokens = current_model.generate(
                            # input_ids = input_tokens['input_ids'],  # Required input IDs
                            # attention_mask = input_tokens['attention_mask'],  # Pass attention mask for longer inputs
                            
                            **input_tokens,  # Automatic device alignment
                            # position_ids=position_ids,
                            # max_new_tokens = 100,
                            max_length = max_length,  # Limit the length of the response
                            do_sample = True,  # Enable sampling for more creative responses
                            pad_token_id = current_tokenizer.eos_token_id  # Handle padding correctly
                        )

        if (print_model_details):
            print("Model Details:\n" + "-"*10)
            print(current_model)
            print("\n" + "-"*100)
        output_to_return = current_tokenizer.decode(output_tokens[0], skip_special_tokens=True)
        
        print(f"Time taken to generate reply = {time.time()- time_start}")
        return f"Reply: \n ------------------\n {output_to_return}\n [END of ANSWER]\n"


## Alternate register_hooks and hook_fn functions

In [ ]:
def register_hooks_alternate(self, layers_to_hook="all"):
    if self.model is None:
        raise ValueError("No model loaded. Please load a model first using load_model(model_name).")
    
    def hook(self, name, module, input, output):
        self.activations[name] = output[0].detach() if isinstance(output, tuple) else output.detach()

    def detect_model_architecture(model):
        if hasattr(model, "model"):
            layers = model.model.layers
            return layers, "up_proj", "down_proj", "self_attn", "mlp"
        elif hasattr(model, "transformer"):
            layers = model.transformer.h
            return layers, "c_fc", "c_proj", "attn", "mlp"
        else:
            raise ValueError("Unknown architecture")

    layers, ffn_in, ffn_out, attn_attr, ffn_module = detect_model_architecture(self.model)
    num_layers = len(layers)
    start_layer = 0 if layers_to_hook == "all" else num_layers - 3

    for layer_idx in range(start_layer, num_layers):
        layer = layers[layer_idx]
        # Use lambda to bind name to the hook call
        getattr(layer, attn_attr).register_forward_hook(lambda m, i, o, name=f"attn_layer{layer_idx}": hook(self, name, m, i, o))
        getattr(getattr(layer, ffn_module), ffn_in).register_forward_hook(lambda m, i, o, name=f"ffn_in_layer{layer_idx}": hook(self, name, m, i, o))
        getattr(getattr(layer, ffn_module), ffn_out).register_forward_hook(lambda m, i, o, name=f"ffn_out_layer{layer_idx}": hook(self, name, m, i, o))

In [89]:
data = {'a': 1, 'b': 2, 'c': 3}
filename = 'data.pkl'

with open(filename, 'wb') as file:
    pickle.dump(data, file)

In [105]:
manager.activation_buffer

{'deepseek-ai/deepseek-coder-1.3b-base': {}}